# Rodada 2 — logit fracional com FE municipal/ano

Mesma amostra do PPML. Frações, inclusive zeros, não são contagens de ensaios. FE não linear sujeito a viés de parâmetros incidentais: diagnóstico de forma funcional, não resultado causal definitivo.

In [1]:
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d
print(len(d),"pares na grade; ver ESTRATEGIA.md para escolhas anteriores à estimação.")

3384 pares na grade; ver ESTRATEGIA.md para escolhas anteriores à estimação.


## 1. Suporte comum

In [2]:
s=d.dropna(subset=['pp_agricultura','pp_pastagem','estoque','expo_soja','expo_milho']).copy()
g=s.groupby('id').agg(n=('ano','size'),ag=('ha_agricultura','sum'),pa=('ha_pastagem','sum'))
validos=g.index[(g.n>1)&(g.ag>0)&(g.pa>0)]
excluidos=g.loc[~g.index.isin(validos)].copy();excluidos.to_csv(e.p/'auditoria_exclusoes_naolinear.csv')
s=s[s.id.isin(validos)].copy()
assert s.estoque.gt(0).all()
for u in ['agricultura','pastagem']:assert s.groupby('ano')['ha_'+u].sum().gt(0).all()
print(len(s),'pares;',s.id.nunique(),'municípios; exclusões explícitas:',len(excluidos))

2769 pares; 134 municípios; exclusões explícitas: 3


## 2. Resposta fracional com heterogeneidade municipal — dois destinos

In [3]:
for uso in ['agricultura','pastagem']:
    e.glm('F01_'+uso,'taxa_'+uso,['expo_soja','expo_milho'],s,tipo='Fracional')
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','se','ic95_inf','ic95_sup','p','N']])
e.salvar('fracional')

,id,termo,coef,se,ic95_inf,ic95_sup,p,N
0,F01_agricultura,expo_soja,0.062133,0.061517,-0.065124,0.189390,0.322993,2769
1,F01_agricultura,expo_milho,0.143223,0.055253,0.028924,0.257522,0.016294,2769


,id,termo,coef,se,ic95_inf,ic95_sup,p,N
2,F01_pastagem,expo_soja,0.066440,0.03035,0.003657,0.129224,0.038995,2769
3,F01_pastagem,expo_milho,0.078294,0.02961,0.017041,0.139547,0.014499,2769
